In [0]:
# 1. Define Widgets (Parameters passed from ADF)
dbutils.widgets.text("landed_path", "")
dbutils.widgets.text("format", "")
dbutils.widgets.text("read_options", "")
dbutils.widgets.text("bronze_table", "")

In [0]:
%run ./generic_audit_notebook

In [0]:
# 2. Get values from Widgets
from datetime import datetime
from pyspark.sql import types as T
from pyspark.sql.functions import current_timestamp, col, lit, to_date, from_json

# ── Timing ────────────────────────────────────────────────────────────────────
run_start_ts = datetime.now()
LOAD_DATE    = run_start_ts.date()

# ── Widget Parameters ─────────────────────────────────────────────────────────
landed_path      = dbutils.widgets.get("landed_path")
fmt_raw          = dbutils.widgets.get("format")          # original (e.g. "tsv")
# Map non-standard format names to Spark-compatible equivalents
FORMAT_ALIASES = {
    "tsv":     "csv",   # tab-separated — use CSV reader with sep=\t
    "txt":     "csv",   # delimited text files
    "tab":     "csv",   # alternate tab-separated label
    "jsonl":   "json",  # JSON lines / newline-delimited JSON
    "ndjson":  "json",  # another NDJSON alias
}
file_format = FORMAT_ALIASES.get(fmt_raw.lower(), fmt_raw.lower())
read_options_str = dbutils.widgets.get("read_options")
bronze_table     = dbutils.widgets.get("bronze_table")    # e.g. "bronze.departments"

# ── Derived Source Metadata ───────────────────────────────────────────────────
source_name    = bronze_table.split(".")[-1]              # "departments"
# full_table     = f"devhcdatabrickspwc.{bronze_table}"        # fully qualified

"""DYNAMIC ENVIRONMENT DETECTION: Inspects the Databricks workspace path to pick dev or prod"""

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    # If the path doesn't contain 'prod', it defaults to 'dev'
    env_prefix = "prod" if "/prod/" in notebook_path.lower() else "dev"
except:
    env_prefix = "dev" # Fallback if run locally/interactively

# Construct the correct catalog dynamically
catalog_name = f"{env_prefix}hcdatabrickspwc"
full_table   = f"{catalog_name}.{bronze_table}"

ingestion_user = spark.sql("SELECT current_user()").collect()[0][0]
run_id         = f"run_{run_start_ts.strftime('%Y%m%d_%H%M%S')}_{source_name}"
batch_id       = str(uuid.uuid4())   # unique per load — shared across bronze rows and audit log

# 3. Parse read_options string into a dictionary
options_dict = {}
if read_options_str and read_options_str.lower() != "null":
    for option in read_options_str.split(";"):
        if "=" in option:
            key, value = option.split("=", 1)
            key = key.strip()
            value = value.strip(' ')   # spaces only — tab/newline are valid delimiters
            # Unescape \t → tab, \n → newline etc. (handles both ADF literal strings and real chars)
            try:
                value = value.encode('raw_unicode_escape').decode('unicode_escape')
            except (UnicodeDecodeError, ValueError):
                pass
            options_dict[key] = value

try:
     # ── 4. Read from Landing csv or tsv 
    if file_format in ["csv", "tsv"]:
        print(f"[INFO] Reading: {landed_path}  |  format: {file_format}")
        df = spark.read.format(file_format).options(**options_dict).load(landed_path)
        records_read = df.count()
        print(f"[INFO] Records read: {records_read:,}")

    # ── 4. Read from landing for json ──────────────────────────────────────────────────
    print(f"[INFO] Reading: {landed_path}  |  format: {file_format}")
    df = spark.read.format(file_format).options(**options_dict).load(landed_path)

    if file_format == "json" and "multiLine" not in options_dict and df.columns == ["_corrupt_record"]:
        print("[INFO] Detected corrupt-only JSON parse; retrying with multiLine=true")
        df = spark.read.format(file_format).options(**options_dict, multiLine="true").load(landed_path)

    records_read = df.count()
    print(f"[INFO] Records read: {records_read:,}")

    if "compensation" in df.columns:
        comp_field = next((f for f in df.schema.fields if f.name == "compensation"), None)
        comp_dt = comp_field.dataType if comp_field else None
        print(f"[INFO] compensation datatype: {comp_dt}")

        if isinstance(comp_dt, T.StructType):
            df = (df
                  .withColumn("salary_amount", col("compensation.salary.amount").cast("double"))
                  .withColumn("salary_currency", col("compensation.salary.currency"))
                  .withColumn("salary_frequency", col("compensation.salary.frequency"))
                  .withColumn("salary_effective_from", to_date(col("compensation.salary.effective_from")))
                  .withColumn("salary_effective_to", to_date(col("compensation.salary.effective_to"))))
        elif isinstance(comp_dt, T.StringType):
            comp_schema = T.StructType([
                T.StructField("salary", T.StructType([
                    T.StructField("effective_to", T.StringType(), True),
                    T.StructField("amount", T.DoubleType(), True),
                    T.StructField("currency", T.StringType(), True),
                    T.StructField("effective_from", T.StringType(), True),
                    T.StructField("frequency", T.StringType(), True),
                ]), True)
            ])
            df = (df
                  .withColumn("compensation_struct", from_json(col("compensation"), comp_schema))
                  .withColumn("salary_amount", col("compensation_struct.salary.amount").cast("double"))
                  .withColumn("salary_currency", col("compensation_struct.salary.currency"))
                  .withColumn("salary_frequency", col("compensation_struct.salary.frequency"))
                  .withColumn("salary_effective_from", to_date(col("compensation_struct.salary.effective_from")))
                  .withColumn("salary_effective_to", to_date(col("compensation_struct.salary.effective_to")))
                  .drop("compensation_struct"))
        else:
            print(f"[WARN] Unexpected compensation type: {comp_dt}. Skipping flattening.")

    if "business_date" in df.columns:
        df = df.withColumn("business_date", to_date(col("business_date")))

    # ── 5. Add Audit Columns ─────────────────────────────────────────────────
    df_with_audit = (df
                     .withColumn("ingestion_timestamp",  current_timestamp())
                     .withColumn("source_file",           col("_metadata.file_path"))
                     .withColumn("_meta_pipeline_name",   lit("pl_landing_to_bronze"))
                     .withColumn("_meta_source_name",     lit(source_name))
                     .withColumn("_meta_batch_id",        lit(batch_id))
                     .withColumn("_meta_run_id",          lit(run_id))
                     .withColumn("_meta_ingestion_user",  lit(ingestion_user))
                     .withColumn("_meta_ingestion_ts",    current_timestamp())
                     .withColumn("_meta_source_file",     col("_metadata.file_path"))
                     .withColumn("_meta_load_time",       current_timestamp())
                     .withColumn("_meta_load_date",       lit(str(LOAD_DATE))))

    # ── 6. Write to Bronze Delta Table ────────────────────────────────────────
    print(f"[INFO] Writing to: {full_table}")
    (df_with_audit.write
     .format("delta")
     .mode("overwrite")
     .option("mergeSchema", "true")
     .saveAsTable(full_table))

    run_end_ts      = datetime.now()
    records_written = records_read    # full overwrite — all records written
    duration_ms     = int((run_end_ts - run_start_ts).total_seconds() * 1000)
    print(f"[INFO] Done. {records_written:,} records written | {duration_ms:,} ms")

    # ── 7. Audit Log — SUCCESS ────────────────────────────────────────────────
    log_audit_ingestion(
        pipeline_name        = "pl_landing_to_bronze",
        layer                = "bronze",
        source_name          = source_name,
        target_table         = full_table,
        source_type          = fmt_raw,
        bronze_table         = bronze_table,
        batch_id             = batch_id,
        run_id               = run_id,
        trigger_type         = "scheduled",
        run_start_ts         = run_start_ts,
        run_end_ts           = run_end_ts,
        last_status          = "SUCCESS",
        records_read         = records_read,
        records_written      = records_written,
        error_count          = 0,
        file_checkpoint_path = landed_path,
        ingestion_user       = ingestion_user,
        last_load_date       = LOAD_DATE,
        notes                = f"ADF ForEach | source: {landed_path}"
    )
    print(f"[AUDIT] SUCCESS logged for {source_name} -> {full_table}")

except Exception as e:
    run_end_ts = datetime.now()
    error_msg  = str(e)
    print(f"[ERROR] Failed: {source_name} — {error_msg[:300]}")

    # ── 7b. Audit Log — FAILED ────────────────────────────────────────────────
    try:
        log_audit_ingestion(
            pipeline_name        = "pl_landing_to_bronze",
            layer                = "bronze",
            source_name          = source_name,
            target_table         = full_table,
            source_type          = fmt_raw,
            bronze_table         = bronze_table,
            run_id               = run_id,
            batch_id             = batch_id,
            trigger_type         = "scheduled",
            run_start_ts         = run_start_ts,
            run_end_ts           = run_end_ts,
            last_status          = "FAILED",
            records_read         = 0,
            records_written      = 0,
            error_count          = 1,
            error_message        = error_msg[:2000],
            file_checkpoint_path = landed_path,
            ingestion_user       = ingestion_user,
            last_load_date       = LOAD_DATE,
            notes                = f"ADF ForEach | source: {landed_path}"
        )
        print(f"[AUDIT] FAILED logged for {source_name}")
    except Exception as audit_err:
        print(f"[WARN] Audit write failed: {audit_err}")

    raise  # Re-raise so ADF marks the ForEach activity as failed

In [ ]:
# In Cell 1 (Define the Parameter)Add a new widget definition line at the bottom of the cell so it can capture the parameter coming from your ADF pipeline.Add at the very end of Cell 1:pythondbutils.widgets.text("load_type", "full_load")  # Default to full_load for safety
# Use code with caution.2. In Cell 3 (Get Value & Implement Write Logic)You need to make two changes in Cell 3: fetch the value near the top, and replace your strict overwrite write block at the bottom.Change A: Retrieve the valueGo to your # ── Widget Parameters ── section (around line 12).Add this line directly below bronze_table = dbutils.widgets.get("bronze_table"):pythonload_type        = dbutils.widgets.get("load_type").lower() # e.g. "full_load" or "incremental_load"
# Use code with caution.Change B: Replace the Write LogicScroll to the bottom of Cell 3 where you see # ── 6. Write to Bronze Delta Table ──.Replace lines 90 to 95:python    # ── 6. Write to Bronze Delta Table ────────────────────────────────────────
#     print(f"[INFO] Writing to: {full_table} using mode: {load_type}")
#     (df_with_audit.write
#      .format("delta")
#      .mode("overwrite")
#      .option("mergeSchema", "true")
#      .saveAsTable(full_table))
# Use code with caution.With this new smart logic:python    # ── 6. Write to Bronze Delta Table ────────────────────────────────────────
#     print(f"[INFO] Writing to: {full_table} using mode: {load_type}")
    
#     if load_type == "incremental_load":
#         # Check if table exists to safely append or merge
#         if spark.catalog.tableExists(full_table):
#             # Using append mode with mergeSchema ensures new fields don't cause pipeline failures
#             (df_with_audit.write
#              .format("delta")
#              .mode("append")
#              .option("mergeSchema", "true")
#              .saveAsTable(full_table))
#         else:
#             # Fallback to overwrite to initialize the table safely if it does not exist yet
#             print(f"[WARN] Table {full_table} does not exist. Initializing with full overwrite.")
#             (df_with_audit.write
#              .format("delta")
#              .mode("overwrite")
#              .option("overwriteSchema", "true")
#              .saveAsTable(full_table))
#     else:
#         # Default behavior: full_load uses complete overwrite
#         (df_with_audit.write
#          .format("delta")
#          .mode("overwrite")
#          .option("overwriteSchema", "true")
#          .saveAsTable(full_table))
# Use code with caution.Important Clean-up NoteLooking closely at your Cell 3, there is a duplicate read block immediately after the csv/tsv logic block:python    if file_format in ["csv", "tsv"]:
#         ...
#         records_read = df.count()
#         print(f"[INFO] Records read: {records_read:,}")

#     # ── 4. Read from landing for json ──────────────────────────────────────────────────
#     print(f"[INFO] Reading: {landed_path}  |  format: {file_format}")
#     df = spark.read.format(file_format).options(**options_dict).load(landed_path)
# Use code with caution.